In [ ]:
from pyspark.sql import functions as F

CATALOG = "spotify_etl"
SCHEMA = "bronze"
TABLE = "bronze_tracks"

dbutils.widgets.text("raw_base_path", "/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw", "RAW base path")
RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")


In [ ]:
import os, json

def collect():
    rows = []
    entity_path = f"{RAW_BASE_PATH}/tracks_bulk"
    if not os.path.exists(entity_path): return rows
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.startswith("page_") and fn.endswith(".json") and not fn.endswith("_meta.json"):
                with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                    data = json.load(f)
                for track in data.get("tracks", []):
                    if not track: continue
                    artists = track.get("artists") or []
                    first = artists[0] if artists else {}
                    album = track.get("album") or {}
                    rows.append({
                        "track_id": track.get("id"), "track_name": track.get("name"),
                        "album_id": album.get("id"), "album_name": album.get("name"),
                        "artist_id": first.get("id"), "artist_name": first.get("name"),
                        "duration_ms": track.get("duration_ms"), "popularity": track.get("popularity"),
                        "explicit": track.get("explicit"), "release_date": album.get("release_date"),
                        "preview_url": track.get("preview_url"),
                    })
    return rows

rows = collect()
if rows:
    df = spark.createDataFrame(rows).dropDuplicates(["track_id"])
    df = df.withColumn("processing_date", F.current_date())
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")
else:
    print(f"No data for {TABLE}")
